In [1]:
from pathlib import Path
import openmm as mm
from openmm import app

ROOT = Path(r"C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns")

system_file = ROOT / "states" / "production_system.xml"
state_file  = ROOT / "states" / "equilibrated.xml"
pdb_file    = ROOT / "structures" / "solvated.pdb"

print("=== FILE ===")
for f in [system_file, state_file, pdb_file]:
    print(f.name, ":", f.exists())

print("\n=== OPENMM ===")
print("Versione:", mm.version.version)

print("\n=== PIATTAFORME DISPONIBILI ===")
for i in range(mm.Platform.getNumPlatforms()):
    p = mm.Platform.getPlatform(i)
    print(i, p.getName())

print("\n=== CARICAMENTO SISTEMA ===")
system = mm.XmlSerializer.deserialize(system_file.read_text())
state = mm.XmlSerializer.deserialize(state_file.read_text())
pdb = app.PDBFile(str(pdb_file))

print("System particles :", system.getNumParticles())
print("State positions  :", len(state.getPositions()))
print("PDB atoms        :", pdb.topology.getNumAtoms())
print("Periodic system  :", system.usesPeriodicBoundaryConditions())

assert system.getNumParticles() == len(state.getPositions())
assert system.getNumParticles() == pdb.topology.getNumAtoms()

print("\nCOMPATIBILITA' FILE: OK")

print("\n=== CUDA ===")
cuda = mm.Platform.getPlatformByName("CUDA")
print("CUDA trovata     :", cuda.getName())
print("CUDA properties  :", cuda.getPropertyNames())

print("\nTEST PRELIMINARE: OK")

=== FILE ===
production_system.xml : True
equilibrated.xml : True
solvated.pdb : True

=== OPENMM ===
Versione: 8.6.1.dev-b399af4

=== PIATTAFORME DISPONIBILI ===
0 Reference
1 CPU
2 CUDA
3 OpenCL

=== CARICAMENTO SISTEMA ===
System particles : 311112
State positions  : 311112
PDB atoms        : 311112
Periodic system  : True

COMPATIBILITA' FILE: OK

=== CUDA ===
CUDA trovata     : CUDA
CUDA properties  : ('DeviceIndex', 'DeviceName', 'UseBlockingSync', 'Precision', 'UseCpuPme', 'CudaCompiler', 'TempDirectory', 'CudaHostCompiler', 'DisablePmeStream', 'DeterministicForces')

TEST PRELIMINARE: OK


In [2]:
import openmm as mm
from openmm import unit

ROOT = Path(r"C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns")

system = mm.XmlSerializer.deserialize(
    (ROOT / "states" / "production_system.xml").read_text()
)

state = mm.XmlSerializer.deserialize(
    (ROOT / "states" / "equilibrated.xml").read_text()
)

integrator = mm.LangevinMiddleIntegrator(
    300 * unit.kelvin,
    1.0 / unit.picosecond,
    0.002 * unit.picoseconds
)

platform = mm.Platform.getPlatformByName("CUDA")

properties = {
    "DeviceIndex": "0",
    "Precision": "mixed"
}

context = mm.Context(system, integrator, platform, properties)
context.setState(state)

check = context.getState(
    getPositions=True,
    getVelocities=True,
    getEnergy=True
)

print("=== CUDA CONTEXT ===")
print("Platform       :", context.getPlatform().getName())
print("DeviceIndex    :", platform.getPropertyValue(context, "DeviceIndex"))
print("DeviceName     :", platform.getPropertyValue(context, "DeviceName"))
print("Precision      :", platform.getPropertyValue(context, "Precision"))

print("\n=== STATE ===")
print("Positions      :", len(check.getPositions()))
print("Velocities     :", len(check.getVelocities()))
print("Potential E    :", check.getPotentialEnergy())
print("Kinetic E      :", check.getKineticEnergy())

print("\nCUDA CONTEXT TEST: OK")

del context
del integrator

OpenMMException: Called setParameter() with invalid parameter name: k

In [3]:
import openmm as mm
from openmm import unit
from pathlib import Path

ROOT = Path(r"C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns")

system = mm.XmlSerializer.deserialize(
    (ROOT / "states" / "production_system.xml").read_text()
)

state = mm.XmlSerializer.deserialize(
    (ROOT / "states" / "equilibrated.xml").read_text()
)

integrator = mm.LangevinMiddleIntegrator(
    300 * unit.kelvin,
    1.0 / unit.picosecond,
    0.002 * unit.picoseconds
)

platform = mm.Platform.getPlatformByName("CUDA")

properties = {
    "DeviceIndex": "0",
    "Precision": "mixed"
}

context = mm.Context(system, integrator, platform, properties)

# Trasferiamo SOLO le grandezze fisiche necessarie.
# Non usiamo context.setState(state), perché lo State di equilibration
# contiene Context parameters non presenti nel production system.
context.setPositions(state.getPositions())
context.setVelocities(state.getVelocities())

a, b, c = state.getPeriodicBoxVectors()
context.setPeriodicBoxVectors(a, b, c)

check = context.getState(
    getPositions=True,
    getVelocities=True,
    getEnergy=True
)

print("=== CUDA CONTEXT ===")
print("Platform       :", context.getPlatform().getName())
print("DeviceIndex    :", platform.getPropertyValue(context, "DeviceIndex"))
print("DeviceName     :", platform.getPropertyValue(context, "DeviceName"))
print("Precision      :", platform.getPropertyValue(context, "Precision"))

print("\n=== STATE ===")
print("Positions      :", len(check.getPositions()))
print("Velocities     :", len(check.getVelocities()))
print("Potential E    :", check.getPotentialEnergy())
print("Kinetic E      :", check.getKineticEnergy())
print("Box vectors    :", check.getPeriodicBoxVectors())

print("\nCUDA CONTEXT TEST: OK")

del context
del integrator

=== CUDA CONTEXT ===
Platform       : CUDA
DeviceIndex    : 0
DeviceName     : NVIDIA GeForce RTX 4090
Precision      : mixed

=== STATE ===
Positions      : 311112
Velocities     : 311112
Potential E    : -4303542.332378019 kJ/mol
Kinetic E      : 785042.6880074306 kJ/mol
Box vectors    : [Vec3(x=14.649706731717345, y=0.0, z=0.0), Vec3(x=0.0, y=14.649706731717345, z=0.0), Vec3(x=0.0, y=0.0, z=14.649706731717345)] nm

CUDA CONTEXT TEST: OK


In [4]:
import time
import openmm as mm
from openmm import unit
from pathlib import Path

ROOT = Path(r"C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns")

system = mm.XmlSerializer.deserialize(
    (ROOT / "states" / "production_system.xml").read_text()
)

state = mm.XmlSerializer.deserialize(
    (ROOT / "states" / "equilibrated.xml").read_text()
)

integrator = mm.LangevinMiddleIntegrator(
    300 * unit.kelvin,
    1.0 / unit.picosecond,
    0.002 * unit.picoseconds
)
integrator.setRandomNumberSeed(20261027)

platform = mm.Platform.getPlatformByName("CUDA")
properties = {
    "DeviceIndex": "0",
    "Precision": "mixed"
}

context = mm.Context(system, integrator, platform, properties)

context.setPositions(state.getPositions())
context.setVelocities(state.getVelocities())

a, b, c = state.getPeriodicBoxVectors()
context.setPeriodicBoxVectors(a, b, c)

# 1000 step × 2 fs = 2 ps
N_STEPS = 1000

print("GPU:", platform.getPropertyValue(context, "DeviceName"))
print("Avvio benchmark di 2 ps...")

t0 = time.time()
integrator.step(N_STEPS)
elapsed = time.time() - t0

simulated_ns = N_STEPS * 0.002 / 1000.0
ns_per_day = simulated_ns / elapsed * 86400.0
estimated_minutes_1ns = 1.0 / ns_per_day * 1440.0

print(f"Tempo reale       : {elapsed:.2f} s")
print(f"Velocità          : {ns_per_day:.2f} ns/day")
print(f"Stima per 1 ns    : {estimated_minutes_1ns:.1f} min")
print("BENCHMARK CUDA: OK")

del context
del integrator

GPU: NVIDIA GeForce RTX 4090
Avvio benchmark di 2 ps...
Tempo reale       : 1.90 s
Velocità          : 90.82 ns/day
Stima per 1 ns    : 15.9 min
BENCHMARK CUDA: OK


In [5]:
from pathlib import Path
from datetime import datetime, timezone

ROOT = Path(r"C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns")

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_ID = f"{timestamp}_seed20261027"

RUN_DIR = ROOT / "production_runs" / RUN_ID

for subdir in ["states", "structures", "trajectories", "tables", "plots"]:
    (RUN_DIR / subdir).mkdir(parents=True, exist_ok=True)

print("RUN_ID :", RUN_ID)
print("RUN_DIR:", RUN_DIR)
print("Directory create:", RUN_DIR.exists())

RUN_ID : 20261027T171745_698045Z_seed20261027
RUN_DIR: C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns\production_runs\20261027T171745_698045Z_seed20261027
Directory create: True


In [6]:
import json
import time
from pathlib import Path
from datetime import datetime, timezone

import openmm as mm
from openmm import app, unit

# ============================================================
# 09l — MOL00484 / EGFR
# OFFICIAL 1 ns CUDA PRODUCTION
# ============================================================

ROOT = Path(r"C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns")

RUN_ID = "20261027T171745_698045Z_seed20261027"
RUN_DIR = ROOT / "production_runs" / RUN_ID

STATE_DIR = RUN_DIR / "states"
TRAJ_DIR  = RUN_DIR / "trajectories"
TABLE_DIR = RUN_DIR / "tables"

for d in [STATE_DIR, TRAJ_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---------- Protocol ----------
SEED = 20261027

TEMPERATURE_K = 300.0
FRICTION_PS   = 1.0
TIMESTEP_PS   = 0.002

PRODUCTION_PS = 1000.0
FRAME_PS      = 2.0

N_STEPS = int(PRODUCTION_PS / TIMESTEP_PS)       # 500000
REPORT_INTERVAL = int(FRAME_PS / TIMESTEP_PS)   # 1000

# Checkpoint every 10 ps
CHECKPOINT_PS = 10.0
CHECKPOINT_INTERVAL = int(CHECKPOINT_PS / TIMESTEP_PS)

print("=== 09l MOL00484 OFFICIAL PRODUCTION ===")
print("RUN_ID              :", RUN_ID)
print("Production          :", PRODUCTION_PS, "ps")
print("Timestep            :", TIMESTEP_PS, "ps")
print("Total steps         :", N_STEPS)
print("Frame interval      :", FRAME_PS, "ps")
print("Expected DCD frames :", N_STEPS // REPORT_INTERVAL)
print("Checkpoint interval :", CHECKPOINT_PS, "ps")

# ============================================================
# LOAD PORTABLE MAC PREPARATION
# ============================================================

system = mm.XmlSerializer.deserialize(
    (ROOT / "states" / "production_system.xml").read_text()
)

eq_state = mm.XmlSerializer.deserialize(
    (ROOT / "states" / "equilibrated.xml").read_text()
)

pdb = app.PDBFile(
    str(ROOT / "structures" / "solvated.pdb")
)

assert system.getNumParticles() == len(eq_state.getPositions())
assert system.getNumParticles() == pdb.topology.getNumAtoms()

print("\nParticles:", system.getNumParticles())

# ============================================================
# INTEGRATOR + CUDA
# ============================================================

integrator = mm.LangevinMiddleIntegrator(
    TEMPERATURE_K * unit.kelvin,
    FRICTION_PS / unit.picosecond,
    TIMESTEP_PS * unit.picoseconds
)

integrator.setRandomNumberSeed(SEED)

platform = mm.Platform.getPlatformByName("CUDA")

properties = {
    "DeviceIndex": "0",
    "Precision": "mixed"
}

simulation = app.Simulation(
    pdb.topology,
    system,
    integrator,
    platform,
    properties
)

# IMPORTANT:
# transfer physical state only; do NOT simulation.context.setState(eq_state)
simulation.context.setPositions(eq_state.getPositions())
simulation.context.setVelocities(eq_state.getVelocities())

a, b, c = eq_state.getPeriodicBoxVectors()
simulation.context.setPeriodicBoxVectors(a, b, c)

print("Platform :", simulation.context.getPlatform().getName())
print(
    "GPU      :",
    platform.getPropertyValue(simulation.context, "DeviceName")
)
print(
    "Precision:",
    platform.getPropertyValue(simulation.context, "Precision")
)

# ============================================================
# INITIAL STATE
# ============================================================

initial_state = simulation.context.getState(
    getPositions=True,
    getVelocities=True,
    getEnergy=True
)

(STATE_DIR / "production_start.xml").write_text(
    mm.XmlSerializer.serialize(initial_state)
)

app.PDBFile.writeFile(
    pdb.topology,
    initial_state.getPositions(),
    open(RUN_DIR / "structures" / "production_start.pdb", "w")
)

print("\nInitial potential energy:",
      initial_state.getPotentialEnergy())
print("Initial kinetic energy  :",
      initial_state.getKineticEnergy())

# ============================================================
# REPORTERS
# ============================================================

dcd_path = TRAJ_DIR / "production_full.dcd"
log_path = TABLE_DIR / "production.csv"
checkpoint_path = STATE_DIR / "production_latest.chk"

simulation.reporters.append(
    app.DCDReporter(
        str(dcd_path),
        REPORT_INTERVAL,
        enforcePeriodicBox=True
    )
)

simulation.reporters.append(
    app.StateDataReporter(
        str(log_path),
        REPORT_INTERVAL,
        step=True,
        time=True,
        potentialEnergy=True,
        kineticEnergy=True,
        totalEnergy=True,
        temperature=True,
        volume=True,
        density=True,
        speed=True,
        separator=","
    )
)

simulation.reporters.append(
    app.CheckpointReporter(
        str(checkpoint_path),
        CHECKPOINT_INTERVAL
    )
)

# ============================================================
# PROVENANCE BEFORE RUN
# ============================================================

provenance = {
    "candidate": "MOL00484",
    "pipeline_stage": "09l",
    "run_type": "OFFICIAL_1NS_PRODUCTION",
    "run_id": RUN_ID,
    "source_state": str(ROOT / "states" / "equilibrated.xml"),
    "source_system": str(ROOT / "states" / "production_system.xml"),
    "source_topology": str(ROOT / "structures" / "solvated.pdb"),
    "platform": "CUDA",
    "device_index": "0",
    "device_name": platform.getPropertyValue(
        simulation.context, "DeviceName"
    ),
    "precision": platform.getPropertyValue(
        simulation.context, "Precision"
    ),
    "temperature_K": TEMPERATURE_K,
    "friction_per_ps": FRICTION_PS,
    "timestep_ps": TIMESTEP_PS,
    "production_ps": PRODUCTION_PS,
    "total_steps": N_STEPS,
    "frame_ps": FRAME_PS,
    "expected_frames": N_STEPS // REPORT_INTERVAL,
    "seed": SEED,
    "started_utc": datetime.now(timezone.utc).isoformat(),
    "note": (
        "Fresh official 1 ns production from the validated "
        "MOL00484 equilibrated state. This is not a continuation "
        "of the 50 ps smoke test or the 2 ps CUDA benchmark."
    )
}

(RUN_DIR / "provenance.json").write_text(
    json.dumps(provenance, indent=2)
)

# ============================================================
# RUN
# ============================================================

print("\nStarting official 1 ns production...")
print("Expected RTX 4090 runtime approximately 16 minutes.\n")

t0 = time.time()

simulation.step(N_STEPS)

elapsed = time.time() - t0

# ============================================================
# FINAL STATE
# ============================================================

final_state = simulation.context.getState(
    getPositions=True,
    getVelocities=True,
    getEnergy=True
)

(STATE_DIR / "production_final.xml").write_text(
    mm.XmlSerializer.serialize(final_state)
)

simulation.saveCheckpoint(
    str(STATE_DIR / "production_final.chk")
)

with open(
    RUN_DIR / "structures" / "production_final.pdb",
    "w"
) as fh:
    app.PDBFile.writeFile(
        pdb.topology,
        final_state.getPositions(),
        fh
    )

# ============================================================
# FINAL METADATA
# ============================================================

elapsed_min = elapsed / 60.0
ns_per_day = 1.0 / elapsed * 86400.0

provenance["finished_utc"] = datetime.now(
    timezone.utc
).isoformat()

provenance["elapsed_seconds"] = elapsed
provenance["elapsed_minutes"] = elapsed_min
provenance["observed_ns_per_day"] = ns_per_day
provenance["final_potential_energy_kJ_mol"] = (
    final_state.getPotentialEnergy()
    .value_in_unit(unit.kilojoule_per_mole)
)
provenance["final_kinetic_energy_kJ_mol"] = (
    final_state.getKineticEnergy()
    .value_in_unit(unit.kilojoule_per_mole)
)

(RUN_DIR / "provenance.json").write_text(
    json.dumps(provenance, indent=2)
)

print("\n========================================")
print("09l OFFICIAL 1 ns PRODUCTION COMPLETED")
print("========================================")
print(f"Elapsed       : {elapsed_min:.2f} min")
print(f"Performance   : {ns_per_day:.2f} ns/day")
print("Final PE      :", final_state.getPotentialEnergy())
print("Final KE      :", final_state.getKineticEnergy())
print("Trajectory    :", dcd_path)
print("Log           :", log_path)
print("Final state   :", STATE_DIR / "production_final.xml")
print("Final chk     :", STATE_DIR / "production_final.chk")
print("RUN_DIR       :", RUN_DIR)

=== 09l MOL00484 OFFICIAL PRODUCTION ===
RUN_ID              : 20261027T171745_698045Z_seed20261027
Production          : 1000.0 ps
Timestep            : 0.002 ps
Total steps         : 500000
Frame interval      : 2.0 ps
Expected DCD frames : 500
Checkpoint interval : 10.0 ps

Particles: 311112
Platform : CUDA
GPU      : NVIDIA GeForce RTX 4090
Precision: mixed

Initial potential energy: -4303542.332085144 kJ/mol
Initial kinetic energy  : 785042.6880074306 kJ/mol

Starting official 1 ns production...
Expected RTX 4090 runtime approximately 16 minutes.


09l OFFICIAL 1 ns PRODUCTION COMPLETED
Elapsed       : 14.75 min
Performance   : 97.61 ns/day
Final PE      : -4305893.205118641 kJ/mol
Final KE      : 784266.373508755 kJ/mol
Trajectory    : C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns\production_runs\20261027T171745_698045Z_seed20261027\trajectories\production_full.dcd
Log           : C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns\production_runs\20261027T171745_698045Z_seed202610

In [ ]:
from pathlib import Path
import mdtraj as md

ROOT = Path(r"C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns")

RUN_DIR = (
    ROOT
    / "production_runs"
    / "20261027T171745_698045Z_seed20261027"
)

DCD = RUN_DIR / "trajectories" / "production_full.dcd"
TOP = ROOT / "structures" / "solvated.pdb"

print("DCD exists :", DCD.exists())
print("TOP exists :", TOP.exists())

traj = md.load(str(DCD), top=str(TOP))

print("\n=== TRAJECTORY ===")
print("Frames :", traj.n_frames)
print("Atoms  :", traj.n_atoms)
print("Time   :", traj.time[0], "→", traj.time[-1], "ps")

assert traj.n_frames == 500
assert traj.n_atoms == 311112

print("\n09l 1 ns TRAJECTORY: OK")

DCD exists : True
TOP exists : True


C:\Users\Roberto\anaconda3\envs\pearl-gpu\Lib\site-packages\mdtraj\formats\pdb\pdbstructure.py:273: UserWarning: Need to guess residue number string    0 starting from residue34996, atom 243860.
  warnings.warn(
C:\Users\Roberto\anaconda3\envs\pearl-gpu\Lib\site-packages\mdtraj\formats\pdb\pdbstructure.py:204: UserWarning: Need to guess atom number   121 starting from atom 280708.
  warnings.warn(


In [ ]:
from pathlib import Path
import mdtraj as md

ROOT = Path(r"C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns")

RUN_DIR = (
    ROOT
    / "production_runs"
    / "20261027T171745_698045Z_seed20261027"
)

DCD = RUN_DIR / "trajectories" / "production_full.dcd"
TOP = ROOT / "structures" / "solvated.pdb"

print("DCD exists :", DCD.exists())
print("DCD size   :", DCD.stat().st_size / (1024**3), "GB")
print("TOP exists :", TOP.exists())

top = md.load_topology(str(TOP))

print("\nTopology atoms:", top.n_atoms)
print("Topology residues:", top.n_residues)

solute_idx = top.select("not water and not element Na and not element Cl")

print("Selected non-water/non-Na/non-Cl atoms:", len(solute_idx))

print("\nLIGHTWEIGHT TOPOLOGY CHECK: OK")

DCD exists : True
DCD size   : 1.7385112829506397 GB
TOP exists : True


C:\Users\Roberto\anaconda3\envs\pearl-gpu\Lib\site-packages\mdtraj\formats\pdb\pdbstructure.py:273: UserWarning: Need to guess residue number string    0 starting from residue34996, atom 243860.
  warnings.warn(
C:\Users\Roberto\anaconda3\envs\pearl-gpu\Lib\site-packages\mdtraj\formats\pdb\pdbstructure.py:204: UserWarning: Need to guess atom number   121 starting from atom 280708.
  warnings.warn(


In [1]:
from pathlib import Path
import os
import psutil

ROOT = Path(r"C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns")
RUN_DIR = ROOT / "production_runs" / "20261027T171745_698045Z_seed20261027"

files = {
    "DCD": RUN_DIR / "trajectories" / "production_full.dcd",
    "PDB solvato": ROOT / "structures" / "solvated.pdb",
    "PDB soluto": ROOT / "structures" / "solute_reference_09f.pdb",
    "atom map": ROOT / "tables" / "solute_atom_map.csv",
}

print("=== FILE ===")
for name, path in files.items():
    print(
        f"{name:12s}",
        "exists =", path.exists(),
        "| size =", round(path.stat().st_size / 1024**2, 2), "MB"
        if path.exists() else ""
    )

mem = psutil.virtual_memory()

print("\n=== MEMORIA WINDOWS ===")
print("RAM totale     :", round(mem.total / 1024**3, 2), "GB")
print("RAM disponibile:", round(mem.available / 1024**3, 2), "GB")
print("RAM usata      :", mem.percent, "%")

process = psutil.Process(os.getpid())
print("Jupyter RAM    :", round(process.memory_info().rss / 1024**3, 3), "GB")

print("\nDIAGNOSTICA LEGGERA: OK")

=== FILE ===
DCD          exists = True | size = 1780.24 MB
PDB solvato  exists = True | size = 24.03 MB
PDB soluto   exists = True | size = 0.72 MB
atom map     exists = True | size = 0.27 MB

=== MEMORIA WINDOWS ===
RAM totale     : 63.14 GB
RAM disponibile: 47.02 GB
RAM usata      : 25.5 %
Jupyter RAM    : 0.055 GB

DIAGNOSTICA LEGGERA: OK


In [2]:
from pathlib import Path
import pandas as pd

ROOT = Path(r"C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns")

MAP = ROOT / "tables" / "solute_atom_map.csv"

df = pd.read_csv(MAP)

print("=== SOLUTE ATOM MAP ===")
print("Rows:", len(df))
print("\nColumns:")
print(list(df.columns))

print("\nFirst 5 rows:")
print(df.head().to_string(index=False))

print("\nLast 5 rows:")
print(df.tail().to_string(index=False))

print("\nSOLUTE MAP CHECK: OK")

=== SOLUTE ATOM MAP ===
Rows: 9332

Columns:
['index', 'name', 'element', 'residue_index', 'residue_name', 'residue_id', 'insertion_code', 'chain', 'component']

First 5 rows:
 index name element  residue_index residue_name  residue_id insertion_code chain component
     0    N       N              0          LEU           1                    B      EGFR
     1    H       H              0          LEU           1                    B      EGFR
     2   H2       H              0          LEU           1                    B      EGFR
     3   H3       H              0          LEU           1                    B      EGFR
     4   CA       C              0          LEU           1                    B      EGFR

Last 5 rows:
 index name element  residue_index residue_name  residue_id insertion_code chain component
  9327  H42       H            614          LIG           1                    L  MOL00484
  9328  H43       H            614          LIG           1                    L  

In [3]:
from pathlib import Path
import mdtraj as md
import pandas as pd

ROOT = Path(r"C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns")
RUN_DIR = ROOT / "production_runs" / "20261027T171745_698045Z_seed20261027"

DCD = RUN_DIR / "trajectories" / "production_full.dcd"
TOP = ROOT / "structures" / "solvated.pdb"
MAP = ROOT / "tables" / "solute_atom_map.csv"

atom_map = pd.read_csv(MAP)
solute_idx = atom_map["index"].to_numpy(dtype=int)

print("Solute atoms:", len(solute_idx))
print("Index range :", solute_idx.min(), "->", solute_idx.max())

print("\nOpening DCD in streaming mode...")

with md.formats.DCDTrajectoryFile(str(DCD), mode="r") as f:
    xyz, cell_lengths, cell_angles = f.read(n_frames=1)

print("Full frame shape :", xyz.shape)
print("Cell lengths     :", cell_lengths)
print("Cell angles      :", cell_angles)

xyz_solute = xyz[:, solute_idx, :]

print("Solute shape     :", xyz_solute.shape)

assert xyz.shape[0] == 1
assert xyz.shape[1] == 311112
assert xyz_solute.shape == (1, 9332, 3)

print("\nONE-FRAME STREAMING TEST: OK")

Solute atoms: 9332
Index range : 0 -> 9331

Opening DCD in streaming mode...
Full frame shape : (1, 311112, 3)
Cell lengths     : [[146.48154 146.48154 146.48154]]
Cell angles      : [[90. 90. 90.]]
Solute shape     : (1, 9332, 3)

ONE-FRAME STREAMING TEST: OK


In [4]:
from pathlib import Path
import mdtraj as md
import pandas as pd
import numpy as np
import time

ROOT = Path(r"C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns")
RUN_DIR = ROOT / "production_runs" / "20261027T171745_698045Z_seed20261027"

DCD_IN = RUN_DIR / "trajectories" / "production_full.dcd"
DCD_OUT = RUN_DIR / "trajectories" / "production_solute.dcd"
MAP = ROOT / "tables" / "solute_atom_map.csv"

atom_map = pd.read_csv(MAP)
solute_idx = atom_map["index"].to_numpy(dtype=int)

assert len(solute_idx) == 9332
assert np.array_equal(solute_idx, np.arange(9332))

print("=== SOLUTE DCD STREAMING EXTRACTION ===")
print("Input :", DCD_IN)
print("Output:", DCD_OUT)
print("Solute atoms:", len(solute_idx))

n_frames = 0
t0 = time.time()

with md.formats.DCDTrajectoryFile(str(DCD_IN), mode="r") as reader, \
     md.formats.DCDTrajectoryFile(str(DCD_OUT), mode="w") as writer:

    while True:
        xyz, cell_lengths, cell_angles = reader.read(n_frames=1)

        if xyz is None or len(xyz) == 0:
            break

        xyz_solute = xyz[:, solute_idx, :]

        writer.write(
            xyz_solute,
            cell_lengths=cell_lengths,
            cell_angles=cell_angles
        )

        n_frames += 1

        if n_frames % 50 == 0:
            print(f"Frames processed: {n_frames}")

elapsed = time.time() - t0

print("\n=== EXTRACTION COMPLETE ===")
print("Frames written :", n_frames)
print("Elapsed        :", round(elapsed, 2), "s")
print("Output exists  :", DCD_OUT.exists())
print("Output size    :", round(DCD_OUT.stat().st_size / 1024**2, 2), "MB")

assert n_frames == 500
assert DCD_OUT.exists()
assert DCD_OUT.stat().st_size > 0

print("\nPRODUCTION_SOLUTE.DCD: OK")

=== SOLUTE DCD STREAMING EXTRACTION ===
Input : C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns\production_runs\20261027T171745_698045Z_seed20261027\trajectories\production_full.dcd
Output: C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns\production_runs\20261027T171745_698045Z_seed20261027\trajectories\production_solute.dcd
Solute atoms: 9332
Frames processed: 50
Frames processed: 100
Frames processed: 150
Frames processed: 200
Frames processed: 250
Frames processed: 300
Frames processed: 350
Frames processed: 400
Frames processed: 450
Frames processed: 500

=== EXTRACTION COMPLETE ===
Frames written : 500
Elapsed        : 0.57 s
Output exists  : True
Output size    : 53.44 MB

PRODUCTION_SOLUTE.DCD: OK


In [5]:
from pathlib import Path
import mdtraj as md
import pandas as pd

ROOT = Path(r"C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns")
RUN_DIR = ROOT / "production_runs" / "20261027T171745_698045Z_seed20261027"

DCD = RUN_DIR / "trajectories" / "production_solute.dcd"
TOP = ROOT / "structures" / "solute_reference_09f.pdb"
MAP = ROOT / "tables" / "solute_atom_map.csv"

atom_map = pd.read_csv(MAP)

print("Loading solute trajectory...")

traj = md.load(str(DCD), top=str(TOP))

print("\n=== SOLUTE TRAJECTORY ===")
print("Frames              :", traj.n_frames)
print("Trajectory atoms    :", traj.n_atoms)
print("Topology atoms      :", traj.topology.n_atoms)
print("Atom-map rows       :", len(atom_map))

assert traj.n_frames == 500
assert traj.n_atoms == 9332
assert traj.topology.n_atoms == 9332
assert len(atom_map) == 9332

print("\n=== COMPONENTS FROM ATOM MAP ===")
print(atom_map["component"].value_counts().to_string())

lig = atom_map[atom_map["component"] == "MOL00484"]
egfr = atom_map[atom_map["component"] == "EGFR"]

print("\nEGFR atoms     :", len(egfr))
print("MOL00484 atoms:", len(lig))

print("\nSOLUTE TRAJECTORY + TOPOLOGY: OK")

Loading solute trajectory...

=== SOLUTE TRAJECTORY ===
Frames              : 500
Trajectory atoms    : 9332
Topology atoms      : 9332
Atom-map rows       : 9332

=== COMPONENTS FROM ATOM MAP ===
component
EGFR        9286
MOL00484      46

EGFR atoms     : 9286
MOL00484 atoms: 46

SOLUTE TRAJECTORY + TOPOLOGY: OK


In [6]:
from pathlib import Path
import numpy as np
import pandas as pd
import mdtraj as md

# ============================================================
# 09l A26b — MOL00484
# Native EGF–EGFR interface trajectory relevance
# Official 1 ns production
# ============================================================

ROOT = Path(r"C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns")
RUN_DIR = ROOT / "production_runs" / "20261027T171745_698045Z_seed20261027"

DCD = RUN_DIR / "trajectories" / "production_solute.dcd"
TOP = ROOT / "structures" / "solute_reference_09f.pdb"
MAP = ROOT / "tables" / "solute_atom_map.csv"

OUT_SUMMARY = RUN_DIR / "tables" / "09l_A26b_EGF_interface_trajectory_relevance.csv"
OUT_OCC = RUN_DIR / "tables" / "09l_A26b_EGF_interface_residue_occupancy.csv"

CONTACT_NM = 0.45
DISCARD_FRACTION = 0.20

EGF_INTERFACE_RESIDUES_B = {
    "ALA415", "ALA68", "ARG29", "ARG353",
    "ASN12", "ASN128", "ASP22", "ASP355",
    "GLN16", "GLN384", "GLN408", "GLN411",
    "GLU90", "GLY18", "GLY441", "HIS346",
    "HIS409", "ILE438", "ILE467", "LEU14",
    "LEU17", "LEU325", "LEU348", "LEU382",
    "LEU69", "LEU98", "LYS13", "LYS465",
    "PHE357", "PHE412", "PRO349", "SER11",
    "SER356", "SER418", "SER440", "SER468",
    "SER99", "THR15", "THR358", "TYR101",
    "TYR45", "VAL350", "VAL417",
}

# ------------------------------------------------------------
# Load trajectory + atom map
# ------------------------------------------------------------

traj = md.load(str(DCD), top=str(TOP))
atom_map = pd.read_csv(MAP)

assert traj.n_frames == 500
assert traj.n_atoms == len(atom_map) == 9332

# ------------------------------------------------------------
# Heavy atoms only
# ------------------------------------------------------------

elements = atom_map["element"].fillna("").astype(str).str.upper()

egfr_mask = (
    (atom_map["component"] == "EGFR") &
    (atom_map["chain"].astype(str) == "B") &
    (elements != "H")
)

lig_mask = (
    (atom_map["component"] == "MOL00484") &
    (elements != "H")
)

# residue key, e.g. ASN12
residue_key = (
    atom_map["residue_name"].astype(str).str.upper()
    + atom_map["residue_id"].astype(int).astype(str)
)

interface_mask = egfr_mask & residue_key.isin(EGF_INTERFACE_RESIDUES_B)

interface_atoms = atom_map.loc[interface_mask, "index"].to_numpy(dtype=int)
ligand_atoms = atom_map.loc[lig_mask, "index"].to_numpy(dtype=int)

found_residues = sorted(set(residue_key[interface_mask]))

print("=== A26b INPUT ===")
print("Trajectory frames             :", traj.n_frames)
print("Native interface expected     :", len(EGF_INTERFACE_RESIDUES_B))
print("Native interface found        :", len(found_residues))
print("Interface heavy atoms         :", len(interface_atoms))
print("MOL00484 heavy atoms          :", len(ligand_atoms))

missing = sorted(EGF_INTERFACE_RESIDUES_B - set(found_residues))
print("Missing interface residues    :", missing)

assert len(found_residues) == 43
assert len(ligand_atoms) > 0

# ------------------------------------------------------------
# Discard first 20%
# ------------------------------------------------------------

discard_n = int(traj.n_frames * DISCARD_FRACTION)
analysis = traj[discard_n:]

print("Discarded frames              :", discard_n)
print("Frames analyzed               :", analysis.n_frames)

assert analysis.n_frames == 400

# ------------------------------------------------------------
# All interface-heavy-atom × ligand-heavy-atom pairs
# ------------------------------------------------------------

pairs = np.array(
    [(i, j) for i in interface_atoms for j in ligand_atoms],
    dtype=np.int32
)

print("Distance pairs                :", len(pairs))

dist = md.compute_distances(
    analysis,
    pairs,
    periodic=True
)

# Minimum interface-ligand distance per frame
frame_min_nm = dist.min(axis=1)

# Any native-interface contact <= 4.5 Å
frame_contact = frame_min_nm <= CONTACT_NM

# ------------------------------------------------------------
# Residue-level occupancy
# ------------------------------------------------------------

occupancy_rows = []

for res in found_residues:

    mask_res = interface_mask & (residue_key == res)
    res_atoms = atom_map.loc[mask_res, "index"].to_numpy(dtype=int)

    res_pairs = np.array(
        [(i, j) for i in res_atoms for j in ligand_atoms],
        dtype=np.int32
    )

    res_dist = md.compute_distances(
        analysis,
        res_pairs,
        periodic=True
    )

    res_min_nm = res_dist.min(axis=1)
    contacted = res_min_nm <= CONTACT_NM

    occupancy_rows.append({
        "residue": res,
        "occupancy": float(contacted.mean()),
        "frames_contacted": int(contacted.sum()),
        "frames_analyzed": int(analysis.n_frames),
        "minimum_distance_A": float(res_min_nm.min() * 10.0),
        "mean_minimum_distance_A": float(res_min_nm.mean() * 10.0),
    })

occ = pd.DataFrame(occupancy_rows)
occ = occ.sort_values(
    ["occupancy", "residue"],
    ascending=[False, True]
).reset_index(drop=True)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

contact_fraction = float(frame_contact.mean())
minimum_A = float(frame_min_nm.min() * 10.0)
mean_A = float(frame_min_nm.mean() * 10.0)
median_A = float(np.median(frame_min_nm) * 10.0)

residues_ever_contacted = int((occ["occupancy"] > 0).sum())
residues_occ50 = int((occ["occupancy"] >= 0.50).sum())

summary = pd.DataFrame([{
    "candidate": "MOL00484",
    "run_id": "20261027T171745_698045Z_seed20261027",
    "production_ps": 1000.0,
    "total_frames": traj.n_frames,
    "discard_fraction": DISCARD_FRACTION,
    "discarded_frames": discard_n,
    "frames_analyzed": analysis.n_frames,
    "contact_cutoff_A": CONTACT_NM * 10.0,
    "native_interface_residues_expected": 43,
    "native_interface_residues_found": len(found_residues),
    "EGF_interface_contact_fraction": contact_fraction,
    "minimum_ligand_interface_distance_A": minimum_A,
    "mean_minimum_distance_A": mean_A,
    "median_minimum_distance_A": median_A,
    "interface_residues_contacted_at_least_once": residues_ever_contacted,
    "interface_residues_occupancy_ge_0_50": residues_occ50,
}])

OUT_SUMMARY.parent.mkdir(parents=True, exist_ok=True)

summary.to_csv(OUT_SUMMARY, index=False)
occ.to_csv(OUT_OCC, index=False)

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("\n=== A26b EGF-INTERFACE TRAJECTORY RELEVANCE ===")
print("Candidate:", "MOL00484")
print(f"EGF-interface contact fraction after discard: {contact_fraction:.6f}")
print(f"Minimum ligand–EGF-interface distance: {minimum_A:.3f} Å")
print(f"Mean minimum distance: {mean_A:.3f} Å")
print(f"Median minimum distance: {median_A:.3f} Å")
print(
    "EGF-interface residues contacted at least once:",
    residues_ever_contacted
)
print(
    "EGF-interface residues with occupancy >= 50%:",
    residues_occ50
)

print("\n=== RESIDUE OCCUPANCY ===")
print(
    occ[occ["occupancy"] > 0][
        ["residue", "occupancy", "frames_contacted",
         "minimum_distance_A", "mean_minimum_distance_A"]
    ].to_string(index=False)
)

print("\nSaved:")
print(OUT_SUMMARY)
print(OUT_OCC)

print("\n09l A26b 1 ns: COMPLETE")

=== A26b INPUT ===
Trajectory frames             : 500
Native interface expected     : 43
Native interface found        : 43
Interface heavy atoms         : 344
MOL00484 heavy atoms          : 25
Missing interface residues    : []
Discarded frames              : 100
Frames analyzed               : 400
Distance pairs                : 8600

=== A26b EGF-INTERFACE TRAJECTORY RELEVANCE ===
Candidate: MOL00484
EGF-interface contact fraction after discard: 1.000000
Minimum ligand–EGF-interface distance: 2.668 Å
Mean minimum distance: 3.222 Å
Median minimum distance: 3.230 Å
EGF-interface residues contacted at least once: 9
EGF-interface residues with occupancy >= 50%: 5

=== RESIDUE OCCUPANCY ===
residue  occupancy  frames_contacted  minimum_distance_A  mean_minimum_distance_A
 HIS409     1.0000               400            2.846914                 3.340482
 HIS346     0.8675               347            3.054560                 3.972116
 GLN408     0.6625               265            2.9913